In [ ]:
print("test")

In [2]:
# %% [markdown]
# # MMLU Dataset Explorer
# Interactive notebook to explore the MMLU dataset by category

# %% [markdown]
## 1. Load the Dataset

# %%
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

# Load the CSV file
df = pd.read_csv('mmlu_full_test.csv')

print(f"Dataset loaded successfully!")
print(f"Total questions: {len(df)}")
print(f"Total categories: {df['Category'].nunique()}")

# %% [markdown]
## 2. Dataset Overview

# %%
# Display category distribution
category_counts = df['Category'].value_counts().sort_index()
print("\nQuestions per category:")
print(category_counts)

# %%
# Show first few rows
print("\nSample data:")
df.head(10)

# %% [markdown]
## 3. Interactive Category Filter

# %%
def display_category_data(category):
    """Display filtered data for selected category"""
    # Filter data
    filtered_df = df[df['Category'] == category].copy()
    
    # Display summary
    print(f"\n{'='*70}")
    print(f"Category: {category.upper()}")
    print(f"{'='*70}")
    print(f"Number of questions: {len(filtered_df)}")
    print(f"\n")
    
    # Display questions in readable format
    for idx, row in filtered_df.iterrows():
        print(f"\nQuestion {row.name + 1}:")
        print(f"Q: {row['Question']}")
        print(f"\nChoices:")
        
        # Parse choices (they're stored as string representation of list)
        import ast
        choices = ast.literal_eval(row['Choices'])
        for i, choice in enumerate(choices):
            print(f"  {chr(65+i)}. {choice}")
        
        print(f"\n✓ Answer: {row['Answer_Letter']}")
        print("-" * 70)
    
    return filtered_df

# Create dropdown widget
category_dropdown = widgets.Dropdown(
    options=sorted(df['Category'].unique()),
    description='Category:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

# Create button
show_button = widgets.Button(
    description='Show Questions',
    button_style='success',
    icon='check'
)

# Output widget
output = widgets.Output()

# Button click handler
def on_button_click(b):
    with output:
        output.clear_output()
        selected_category = category_dropdown.value
        display_category_data(selected_category)

show_button.on_click(on_button_click)

# Display widgets
print("Select a category and click 'Show Questions':")
display(widgets.VBox([category_dropdown, show_button, output]))

# %% [markdown]
## 4. Search Questions by Keyword

# %%
def search_questions(keyword):
    """Search for questions containing keyword"""
    mask = df['Question'].str.contains(keyword, case=False, na=False)
    results = df[mask]
    
    print(f"\nFound {len(results)} questions containing '{keyword}':")
    print(f"\nCategories: {results['Category'].unique()}")
    
    return results

# Search widget
search_text = widgets.Text(
    placeholder='Enter keyword to search',
    description='Search:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

search_button = widgets.Button(
    description='Search',
    button_style='info',
    icon='search'
)

search_output = widgets.Output()

def on_search_click(b):
    with search_output:
        search_output.clear_output()
        if search_text.value:
            results = search_questions(search_text.value)
            display(results[['Category', 'Question', 'Answer_Letter']])

search_button.on_click(on_search_click)

display(widgets.VBox([search_text, search_button, search_output]))

# %% [markdown]
## 5. Quick Analysis Functions

# %%
def get_category_summary(category):
    """Get summary statistics for a category"""
    cat_df = df[df['Category'] == category]
    
    summary = {
        'Category': category,
        'Total Questions': len(cat_df),
        'Answer Distribution': cat_df['Answer_Letter'].value_counts().to_dict()
    }
    
    return summary

# Example: Get summary for clinical knowledge
summary = get_category_summary('clinical_knowledge')
print(summary)

# %%
def export_category(category, filename=None):
    """Export a specific category to CSV"""
    cat_df = df[df['Category'] == category]
    
    if filename is None:
        filename = f"mmlu_{category}.csv"
    
    cat_df.to_csv(filename, index=False)
    print(f"Exported {len(cat_df)} questions to {filename}")

# Example: Export a category
# export_category('clinical_knowledge', 'clinical_questions.csv')

# %% [markdown]
## 6. Compare Multiple Categories

# %%
def compare_categories(categories):
    """Compare question counts across multiple categories"""
    comparison = df[df['Category'].isin(categories)].groupby('Category').size()
    
    print("\nCategory Comparison:")
    print(comparison)
    
    # Optional: Create a simple bar chart if matplotlib is available
    try:
        import matplotlib.pyplot as plt
        comparison.plot(kind='bar', figsize=(10, 5))
        plt.title('Questions per Category')
        plt.ylabel('Number of Questions')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("\nInstall matplotlib for visualization: pip install matplotlib")
    
    return comparison

# Example: Compare STEM categories
# stem_categories = ['high_school_physics', 'high_school_chemistry', 'high_school_biology']
# compare_categories(stem_categories)

# %% [markdown]
## 7. Random Question Selector

# %%
def get_random_questions(category=None, n=5):
    """Get n random questions, optionally from a specific category"""
    if category:
        sample_df = df[df['Category'] == category].sample(n=min(n, len(df[df['Category'] == category])))
    else:
        sample_df = df.sample(n=min(n, len(df)))
    
    print(f"\nRandom Questions ({len(sample_df)}):")
    
    for idx, row in sample_df.iterrows():
        print(f"\n{'='*70}")
        print(f"Category: {row['Category']}")
        print(f"Q: {row['Question']}")
        
        import ast
        choices = ast.literal_eval(row['Choices'])
        for i, choice in enumerate(choices):
            print(f"  {chr(65+i)}. {choice}")
        
        print(f"\n✓ Answer: {row['Answer_Letter']}")
    
    return sample_df

# Example: Get 3 random questions from any category
# get_random_questions(n=3)

# Example: Get 5 random questions from clinical_knowledge
# get_random_questions(category='clinical_knowledge', n=5)

# %%

Dataset loaded successfully!
Total questions: 14042
Total categories: 57

Questions per category:
Category
abstract_algebra                        100
anatomy                                 135
astronomy                               152
business_ethics                         100
clinical_knowledge                      265
college_biology                         144
college_chemistry                       100
college_computer_science                100
college_mathematics                     100
college_medicine                        173
college_physics                         102
computer_security                       100
conceptual_physics                      235
econometrics                            114
electrical_engineering                  145
elementary_mathematics                  378
formal_logic                            126
global_facts                            100
high_school_biology                     310
high_school_chemistry                   203
high_school_c

{'Category': 'clinical_knowledge', 'Total Questions': 265, 'Answer Distribution': {'D': 79, 'B': 71, 'C': 58, 'A': 57}}


In [ ]:
print("test")
#get_random_questions(n=3)